## How to use torchdiff !

In [2]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from torchdiff.ldm import AutoencoderLDM, TrainAE, TrainLDM, SampleLDM
from torchdiff.ddpm import ForwardDDPM, ReverseDDPM, HyperParamsDDPM, TrainDDPM, SampleDDPM
from torchdiff.ddim import ForwardDDIM, ReverseDDIM, HyperParamsDDIM, TrainDDIM, SampleDDIM
from torchdiff.sde import ForwardSDE, ReverseSDE, HyperParamsSDE, TrainSDE, SampleSDE
from torchdiff.utils import TextEncoder, NoisePredictor, Metrics

### Data Preparation

In [4]:
# Transform: convert to tensor and normalize (mean=0.5, std=0.5 for grayscale images)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Load FashionMNIST (black and white, 28x28)
train_dataset = datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

# Define subset sizes (e.g. 1000 samples from training, 200 from test)
train_subset_indices = torch.randperm(len(train_dataset))[:50]
test_subset_indices = torch.randperm(len(test_dataset))[:10]

# Create subsets
train_subset = Subset(train_dataset, train_subset_indices)
test_subset = Subset(test_dataset, test_subset_indices)

# Create DataLoaders from subsets
tr = DataLoader(train_subset, batch_size=10, shuffle=True)
te = DataLoader(test_subset, batch_size=10, shuffle=False)

#### Training and Sampling process of DDPM

In [6]:
# noise predictor
noise_p = NoisePredictor(
        in_channels=1,
        down_channels=[16, 32],
        mid_channels=[32, 32],
        up_channels=[32, 16],
        down_sampling=[True, True],
        time_embed_dim=32,
        y_embed_dim=32,
        num_down_blocks=2,
        num_mid_blocks=2,
        num_up_blocks=2,
        down_sampling_factor=2
)
# prompt conditional model
cond = TextEncoder(
        use_pretrained_model=True,
        model_name="bert-base-uncased",
        vocabulary_size=30522,
        num_layers=2,
        input_dimension=32,
        output_dimension=32,
        num_heads=2,
        context_length=77
)
# ddpm hyper parameter model
hp_ddpm = HyperParamsDDPM(num_steps=500, beta_start=1e-4, beta_end=0.02, beta_method="linear")

# forward ddpm
f_ddpm = reverse = ForwardDDPM(hp_ddpm)

# reverse ddpm
r_ddpm = ReverseDDPM(hp_ddpm)

# optimizer
opt = torch.optim.Adam(
    [p for p in noise_p.parameters() if p.requires_grad] +
    [p for p in cond.parameters() if p.requires_grad], lr=1e-3
)

# loss function
obj = nn.MSELoss()

# metrics computation
met = Metrics(device="cuda", fid=True, metrics=True, lpips_=True)

# train ddpm model
t_ddpm = TrainDDPM(
    noise_predictor=noise_p,
    hyper_params=hp_ddpm,
    conditional_model=None, #cond,
    metrics_=None, # met,
    optimizer=opt,
    objective=obj,
    data_loader=tr,
    val_loader=None, #te,
    max_epoch=5,
    device="cuda",
    store_path="test_ddpm.pth",
    val_frequency=1
)

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /home/loqman/Downloads/projs/.venv/lib/python3.12/site-packages/lpips/weights/v0.1/vgg.pth


In [7]:
# train process
t_ddpm()

/home/loqman/Downloads/projs/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:227: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  4.63it/s]



Epoch: 1 | Train Loss: 1.5079
Model saved at epoch 1


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00,  7.02it/s]



Epoch: 2 | Train Loss: 1.4151
Model saved at epoch 2


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00,  7.02it/s]


...

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00,  6.86it/s]



Epoch: 4 | Train Loss: 1.1055
Model saved at epoch 4


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00,  7.03it/s]


Epoch: 5 | Train Loss: 0.9366
Model saved at epoch 5


([1.507897138595581,
  1.4150829315185547,
  1.2810924053192139,
  1.1055386066436768,
  0.936623752117157],
 0.936623752117157)

In [8]:
# uploade from checkpoint
t_ddpm.load_checkpoint("test_ddpm.pth")

Loaded checkpoint from test_ddpm.pth at epoch 5 with loss 0.9366


(5, 0.936623752117157)

In [9]:
t_ddpm()

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00,  6.27it/s]



Epoch: 1 | Train Loss: 0.7657
Model saved at epoch 1


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00,  6.94it/s]



Epoch: 2 | Train Loss: 0.6501
Model saved at epoch 2


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00,  6.93it/s]


...

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00,  6.97it/s]



Epoch: 4 | Train Loss: 0.4868
Model saved at epoch 4


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00,  6.69it/s]


Epoch: 5 | Train Loss: 0.4658
Model saved at epoch 5


([0.7656975984573364,
  0.6500636339187622,
  0.5176534056663513,
  0.48684048652648926,
  0.46580368280410767],
 0.46580368280410767)

In [10]:
# sampling process
sampler = SampleDDPM(
    reverse_diffusion=r_ddpm,
    noise_predictor=noise_p,
    image_shape=(28, 28),
    conditional_model=cond,
    tokenizer="bert-base-uncased",
    max_length=77,
    batch_size=3,
    in_channels=1,
    device="cuda",
    output_range=(-1, 1))

In [11]:
samp = sampler(conditions=["a cat", "a dog", "a box"], save_images=True, save_path="ddpm_generated")

#### Training and Sampling process of DDIM model

In [12]:
# noise predictor
noise_p = NoisePredictor(
        in_channels=1,
        down_channels=[16, 32, 64],
        mid_channels=[64, 64],
        up_channels=[64, 32, 16],
        down_sampling=[True, True, True],
        time_embed_dim=64,
        y_embed_dim=64,
        num_down_blocks=2,
        num_mid_blocks=2,
        num_up_blocks=2,
        down_sampling_factor=2
)
# prompt conditional model
cond = TextEncoder(
        use_pretrained_model=True,
        model_name="bert-base-uncased",
        vocabulary_size=30522,
        num_layers=2,
        input_dimension=64,
        output_dimension=64,
        num_heads=2,
        context_length=77
)
# ddim hyper parameter model
hp_ddim = HyperParamsDDIM(num_steps=500, tau_num_steps=100, beta_start=1e-4, beta_end=0.02, beta_method="linear")

# forward ddim
f_ddim = ForwardDDIM(hp_ddim)

# reverse ddim
r_ddim = ReverseDDIM(hp_ddim)

# optimizer
opt = torch.optim.Adam(
    [p for p in noise_p.parameters() if p.requires_grad] +
    [p for p in cond.parameters() if p.requires_grad], lr=1e-3
)

# loss function
obj = nn.MSELoss()

# metrics computation
met = Metrics(device="cuda", fid=True, metrics=True, lpips_=True)

# train ddpm""""""
t_ddim = TrainDDIM(
    noise_predictor=noise_p,
    hyper_params=hp_ddim,
    conditional_model=cond,
    metrics_=met,
    optimizer=opt,
    objective=obj,
    data_loader=tr,
    val_loader=te,
    max_epoch=5,
    device="cuda",
    store_path="test_ddim.pth",
    val_frequency=3
)

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /home/loqman/Downloads/projs/.venv/lib/python3.12/site-packages/lpips/weights/v0.1/vgg.pth


In [13]:
t_ddim()

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  4.96it/s]



Epoch: 1 | Train Loss: 1.6263


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00,  5.31it/s]



Epoch: 2 | Train Loss: 1.4475


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00,  5.01it/s]


...

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  4.96it/s]



Epoch: 4 | Train Loss: 1.0734


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  4.75it/s]


Epoch: 5 | Train Loss: 0.9100


([1.6263267993927002,
  1.4474589824676514,
  1.2404175996780396,
  1.073443055152893,
  0.9099725484848022],
 1.044525384902954)

In [14]:
sampler_ddim = SampleDDIM(
    reverse_diffusion=r_ddim,
    noise_predictor=noise_p,
    image_shape=(64, 64),
    conditional_model=cond,
    tokenizer="bert-base-uncased",
    max_length=77,
    batch_size=5,
    in_channels=1,
    device="cuda",
    output_range=(-1, 1))

In [15]:
gen = sampler_ddim(
    conditions=["nothing", "something", "a cat is playing with a dog", "a rainy day", "some firends talking to each other"],
    save_images=True,
    save_path="ddim_generated"
)

#### Training and Sampling process of SDE models

In [16]:
# noise predictor
noise_p = NoisePredictor(
        in_channels=1,
        down_channels=[8, 16, 32],
        mid_channels=[32, 32, 32],
        up_channels=[32, 16, 8],
        down_sampling=[True, True, True],
        time_embed_dim=32,
        y_embed_dim=32,
        num_down_blocks=1,
        num_mid_blocks=1,
        num_up_blocks=1,
        down_sampling_factor=1
)
# prompt conditional model
cond = TextEncoder(
        use_pretrained_model=True,
        model_name="bert-base-uncased",
        vocabulary_size=30522,
        num_layers=2,
        input_dimension=32,
        output_dimension=32,
        num_heads=1,
        context_length=77
)
# sde hyper parameter model
hp_sde = HyperParamsSDE(
    num_steps=500, beta_start=1e-4, beta_end=0.02,
    sigma_start=1e-3, sigma_end=10.0, start=0.0,
    end=1.0, beta_method="linear"
)

# forward sde
f_sde = ForwardSDE(hp_sde, "ode")

# reverse sde
r_sde = ReverseSDE(hp_sde, "ode")

# optimizer
opt = torch.optim.Adam(
    [p for p in noise_p.parameters() if p.requires_grad] +
    [p for p in cond.parameters() if p.requires_grad], lr=1e-3
)

# loss function
obj = nn.MSELoss()

# metrics computation
met = Metrics(device="cuda", fid=True, metrics=True, lpips_=True)

# train sde
t_sde = TrainSDE(
    method="ode", # "ve", "vp", "sub-vp", "ode"
    noise_predictor=noise_p,
    hyper_params=hp_sde,
    conditional_model=cond,
    metrics_=met,
    optimizer=opt,
    objective=obj,
    data_loader=tr,
    val_loader=te,
    max_epoch=5,
    device="cuda",
    store_path="test_sde.pth",
    val_frequency=3
)

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /home/loqman/Downloads/projs/.venv/lib/python3.12/site-packages/lpips/weights/v0.1/vgg.pth


In [17]:
t_sde()

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  3.99it/s]



Epoch: 1 | Train Loss: 1.3724


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  4.22it/s]



Epoch: 2 | Train Loss: 1.3545


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  4.34it/s]


...

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  4.28it/s]



Epoch: 4 | Train Loss: 1.2506


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  4.19it/s]


Epoch: 5 | Train Loss: 1.2269


([1.3724429607391357,
  1.3544800281524658,
  1.278637170791626,
  1.2506014108657837,
  1.2268511056900024],
 1.185789704322815)

In [18]:
sampler_sde = SampleSDE(
    reverse_diffusion=r_sde,
    noise_predictor=noise_p,
    image_shape=(28, 28),
    conditional_model=None, # cond,
    tokenizer="bert-base-uncased",
    max_length=77,
    batch_size=3,
    in_channels=1,
    device="cuda",
    output_range=(-1, 1))

In [19]:
imgs = sampler_sde(save_images=True, save_path="sde_generated")

#### Training and Sampling of LDM models

In [20]:
# auto-encoder to bring images into latent space
comp = AutoencoderLDM(
        in_channels=1,
        down_channels=[8, 16],
        up_channels=[16, 8],
        out_channels=1,
        dropout_rate=.2,
        latent_channels=1,
        num_heads=1,
        num_groups=8,
        num_layers_per_block=1,
        total_down_sampling_factor=2,
        num_embeddings=16
    )

# noise predictor
noise_p = NoisePredictor(
        in_channels=1,
        down_channels=[16, 32],
        mid_channels=[32, 32],
        up_channels=[32, 16],
        down_sampling=[True, True],
        time_embed_dim=32,
        y_embed_dim=32,
        num_down_blocks=1,
        num_mid_blocks=1,
        num_up_blocks=1,
        down_sampling_factor=2
)
# prompt conditional model
cond = TextEncoder(
        use_pretrained_model=True,
        model_name="bert-base-uncased",
        vocabulary_size=30522,
        num_layers=2,
        input_dimension=32,
        output_dimension=32,
        num_heads=2,
        context_length=77
)
# ddpm hyper parameter model
hp_sde = HyperParamsSDE(
    num_steps=500, beta_start=1e-4, beta_end=0.02,
    sigma_start=1e-3, sigma_end=10.0, start=0.0,
    end=1.0, beta_method="linear"
)

# forward sde
f_sde = ForwardSDE(hp_sde, "ode")

# reverse sde
r_sde = ReverseSDE(hp_sde, "ode")

# optimizer
opt = torch.optim.Adam(
    [p for p in noise_p.parameters() if p.requires_grad] +
    [p for p in cond.parameters() if p.requires_grad], lr=1e-3
)

# loss function
obj = nn.MSELoss()

# metrics computation
met = Metrics(device="cuda", fid=True, metrics=True, lpips_=True)

t_ldm = TrainLDM(
    model="sde", # "ddpm", "ddim", "sde"
    forward_model=f_sde,
    noise_predictor=noise_p,
    hyper_params=hp_sde,
    compressor_model=comp,
    conditional_model=None, #cond,
    reverse_diffusion=r_sde,
    metrics_=None, #met,
    optimizer=opt,
    objective=obj,
    data_loader=tr,
    val_loader=te,
    max_epoch=5,
    device="cuda",
    store_path="test_ldm.pth",
    val_frequency=3
)

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /home/loqman/Downloads/projs/.venv/lib/python3.12/site-packages/lpips/weights/v0.1/vgg.pth


In [21]:
t_ldm()

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 36.14it/s]


Epoch: 1 | Train Loss: 2.7112


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 51.90it/s]


Epoch: 2 | Train Loss: 1.8648


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 53.99it/s]


...

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 53.00it/s]


Epoch: 4 | Train Loss: 1.3733


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 55.01it/s]

Epoch: 5 | Train Loss: 1.1343


([2.711179256439209,
  1.8647531270980835,
  1.6501023769378662,
  1.3733395338058472,
  1.1343176364898682],
 1.5544154644012451)

In [22]:
sampler_ldm = SampleLDM(
    model="sde",
    reverse_diffusion=r_sde,
    noise_predictor=noise_p,
    compressor_model=comp,
    image_shape=(64, 64),
    conditional_model=cond,
    tokenizer="bert-base-uncased",
    max_length=77,
    batch_size=3,
    in_channels=1,
    device="cuda",
    output_range=(-1, 1))

imgs = sampler_ldm(conditions=["nothing", "something", "a cat is playing with a dog"], save_images=True, save_path="ldm_generated")

#### Train ldm autoencoder

In [23]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Load FashionMNIST (black and white, 28x28)
train_dataset = datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

# Define subset sizes (e.g. 1000 samples from training, 200 from test)
train_subset_indices = torch.randperm(len(train_dataset))[:40]
test_subset_indices = torch.randperm(len(test_dataset))[:10]

# Create subsets
train_subset = Subset(train_dataset, train_subset_indices)
test_subset = Subset(test_dataset, test_subset_indices)

# Create DataLoaders from subsets
tr = DataLoader(train_subset, batch_size=20, shuffle=True)
te = DataLoader(test_subset, batch_size=10, shuffle=False)


# train auto-encoder of ldm models
comp = AutoencoderLDM(
    in_channels=1,
    down_channels=[8, 16, 32],
    up_channels=[32, 16, 8],
    out_channels=1,
    dropout_rate=.2,
    latent_channels=1,
    num_heads=2,
    num_groups=8,
    num_layers_per_block=2,
    total_down_sampling_factor=2,
    num_embeddings=16
)
met = Metrics(device="cuda", fid=True, metrics=True, lpips_=True)

opt = torch.optim.Adam(comp.parameters(), lr=1e-3)
# loss function
obj = nn.MSELoss()

t_auto = TrainAE(
    model=comp,
    optimizer=opt,
    data_loader=tr,
    val_loader=te,
    max_epoch=5,
    metrics_=met,
    device="cuda",
    save_path="vlc_model.pth",
    checkpoint=3,
    kl_warmup_epochs=2,
    patience=5,
    val_frequency=2
)

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /home/loqman/Downloads/projs/.venv/lib/python3.12/site-packages/lpips/weights/v0.1/vgg.pth


In [24]:
t_auto()

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  4.29it/s]


Epoch: 1 | Train Loss: 0.7853 | Model saved at epoch 1


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  5.56it/s]


Epoch: 2 | Train Loss: 0.6887Warning: batch size is bigger than the data size. Setting batch size to data size


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  7.12it/s]


...

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  7.11it/s]


 | Val Loss: 0.4301 | FID: 329.4321 | MSE: 0.4002 | PSNR: 3.9769 | SSIM: 0.1947 | LPIPS: 0.4787
 | Model saved at epoch 4


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  4.74it/s]

Epoch: 5 | Train Loss: 0.4683

([0.7853401899337769,
  0.6887050867080688,
  0.5963854789733887,
  0.5262686610221863,
  0.4683226943016052],
 0.4301048219203949)